# Studio post-fix — Embedding NON_PROSE_ORIGINS

Analisi **post-fix** dello stato di `embedded` per `raw_documents` dopo l'introduzione
di `NON_PROSE_ORIGINS = ("gdelt", "comtrade")` in `pathosphere/semantic/embedder.py`:
`get_unembedded_documents` ora esclude questi origin dalla query di selezione
(`WHERE embedded=0 AND origin NOT IN NON_PROSE_ORIGINS`), impedendo che i titoli
sintetici GDELT (`"GDELT: ACTOR1 → ACTOR2 [CODE]"`, non prosa reale) vengano
imbarcati nell'embedder e5-small.

Il fix è **non retroattivo**: agisce solo sulla selezione di NUOVI documenti da
embeddare, non tocca i documenti già marcati `embedded=1` prima del fix (contaminazione
storica pre-esistente in `vec_documents`).

Riferimento codice: `pathosphere/semantic/embedder.py`.

In [1]:
import sys, struct, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pathosphere").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from pathosphere.db.schema import get_connection

DB_PATH = REPO_ROOT / "data" / "db" / "pathosphere.db"
conn = get_connection(DB_PATH)
print(f"DB: {DB_PATH}  exists={DB_PATH.exists()}")

def q(sql, params=()):
    return pd.read_sql_query(sql, conn, params=params)

q("SELECT COUNT(*) AS raw_documents FROM raw_documents")


DB: /Users/dom/Documents/GitHub/pathosphere/data/db/pathosphere.db  exists=True


,raw_documents
0,180505


## 1. Stato `embedded` per origin (query fresca)

In [2]:
embed_by_origin = q("""
SELECT origin, embedded, COUNT(*) AS n
FROM raw_documents
GROUP BY origin, embedded
ORDER BY origin, embedded
""")
embed_by_origin


,origin,embedded,n
0,comtrade,1,252
1,gdelt,0,2995
2,gdelt,1,174286
3,rss,1,2972


In [3]:
pivot = embed_by_origin.pivot_table(index="origin", columns="embedded", values="n", fill_value=0)
pivot.columns = [f"embedded={c}" for c in pivot.columns]
pivot["total"] = pivot.sum(axis=1)
pivot


,embedded=0,embedded=1,total
origin,,,
comtrade,0.0,252.0,252.0
gdelt,2995.0,174286.0,177281.0
rss,0.0,2972.0,2972.0


## 2. Contaminazione storica vs correttezza post-fix

Per `origin in ('gdelt', 'comtrade')`:
- `embedded=1` → contaminazione STORICA (embeddati PRIMA del fix, il fix non li tocca retroattivamente)
- `embedded=0` → CORRETTO, mai passati all'embedder (fix funziona per i nuovi/non ancora processati)

In [4]:
NON_PROSE = ("gdelt", "comtrade")
contam = q(f"""
SELECT origin,
       SUM(CASE WHEN embedded = 1 THEN 1 ELSE 0 END) AS embedded_1_contaminazione_storica,
       SUM(CASE WHEN embedded = 0 THEN 1 ELSE 0 END) AS embedded_0_corretto,
       COUNT(*) AS totale
FROM raw_documents
WHERE origin IN ({",".join("?" for _ in NON_PROSE)})
GROUP BY origin
""", params=NON_PROSE)
contam


,origin,embedded_1_contaminazione_storica,embedded_0_corretto,totale
0,comtrade,252,0,252
1,gdelt,174286,2995,177281


In [5]:
for _, r in contam.iterrows():
    tot = int(r["totale"])
    c1 = int(r["embedded_1_contaminazione_storica"])
    c0 = int(r["embedded_0_corretto"])
    pct = c1 / tot * 100 if tot else 0.0
    print(f"origin={r['origin']}: {c1}/{tot} ({pct:.1f}%) embedded=1 (contaminazione storica pre-fix) | "
          f"{c0}/{tot} ({c0/tot*100:.1f}%) embedded=0 (corretto, mai processati)")

tot_contam = int(contam["embedded_1_contaminazione_storica"].sum())
tot_all_non_prose = int(contam["totale"].sum())
print()
print(f"TOTALE NON_PROSE_ORIGINS: {tot_contam}/{tot_all_non_prose} "
      f"({tot_contam/tot_all_non_prose*100:.1f}%) documenti gdelt/comtrade contaminano ancora "
      "vec_documents/embeddings — il fix non li ha rimossi, solo bloccato i futuri.")


origin=comtrade: 252/252 (100.0%) embedded=1 (contaminazione storica pre-fix) | 0/252 (0.0%) embedded=0 (corretto, mai processati)
origin=gdelt: 174286/177281 (98.3%) embedded=1 (contaminazione storica pre-fix) | 2995/177281 (1.7%) embedded=0 (corretto, mai processati)

TOTALE NON_PROSE_ORIGINS: 174538/177533 (98.3%) documenti gdelt/comtrade contaminano ancora vec_documents/embeddings — il fix non li ha rimossi, solo bloccato i futuri.


## 3. Confronto con `rss` (baseline "sano")

Per contestualizzare: quota di `rss` embedded (dovrebbe essere ~100%, sono prosa reale
e nessun filtro li esclude).

In [6]:
rss_state = q("""
SELECT embedded, COUNT(*) AS n FROM raw_documents WHERE origin = 'rss' GROUP BY embedded
""")
rss_state


,embedded,n
0,1,2972


## 4. Vettori realmente presenti in `vec_documents` per origin

`vec_documents` ha PK `document_id` (join diretto con `raw_documents`, non serve
tabella ausiliaria — verificato con `PRAGMA table_info(vec_documents)`).

In [7]:
vec_cols = q("PRAGMA table_info(vec_documents)")
print("colonne vec_documents:")
vec_cols


colonne vec_documents:


,cid,name,type,notnull,dflt_value,pk
0,0,document_id,,1,None,1
1,1,embedding,,0,None,0


In [8]:
vec_total = int(q("SELECT COUNT(*) AS n FROM vec_documents")["n"][0])
print(f"vec_documents totale righe (vettori memorizzati): {vec_total}")


vec_documents totale righe (vettori memorizzati): 171768


In [9]:
vec_by_origin = q("""
SELECT rd.origin, COUNT(*) AS n_vettori
FROM vec_documents vd
JOIN raw_documents rd ON rd.id = vd.document_id
GROUP BY rd.origin
ORDER BY n_vettori DESC
""")
vec_by_origin


,origin,n_vettori
0,gdelt,168544
1,rss,2972
2,comtrade,252


In [10]:
vec_gdelt = int(vec_by_origin.loc[vec_by_origin["origin"] == "gdelt", "n_vettori"].sum()) if "gdelt" in vec_by_origin["origin"].values else 0
vec_comtrade = int(vec_by_origin.loc[vec_by_origin["origin"] == "comtrade", "n_vettori"].sum()) if "comtrade" in vec_by_origin["origin"].values else 0
vec_rss = int(vec_by_origin.loc[vec_by_origin["origin"] == "rss", "n_vettori"].sum()) if "rss" in vec_by_origin["origin"].values else 0

print(f"Vettori reali in vec_documents per origin=gdelt: {vec_gdelt}")
print(f"Vettori reali in vec_documents per origin=comtrade: {vec_comtrade}")
print(f"Vettori reali in vec_documents per origin=rss: {vec_rss}")
print()
pct_gdelt_of_total = vec_gdelt / vec_total * 100 if vec_total else 0
print(f"I vettori 'gdelt' rappresentano il {pct_gdelt_of_total:.1f}% di tutti i vettori in vec_documents "
      f"({vec_gdelt}/{vec_total}) — spazio indice occupato da titoli sintetici non-prosa, "
      "mai utile per ricerca semantica reale.")


Vettori reali in vec_documents per origin=gdelt: 168544
Vettori reali in vec_documents per origin=comtrade: 252
Vettori reali in vec_documents per origin=rss: 2972

I vettori 'gdelt' rappresentano il 98.1% di tutti i vettori in vec_documents (168544/171768) — spazio indice occupato da titoli sintetici non-prosa, mai utile per ricerca semantica reale.


## 5. Coerenza: `embedded=1` in raw_documents vs righe reali in vec_documents

Verifica se `embedded=1` corrisponde davvero a un vettore scritto (nessun drift tra il
flag e la tabella vettoriale).

In [11]:
coerenza = q("""
SELECT rd.origin,
       SUM(CASE WHEN rd.embedded = 1 THEN 1 ELSE 0 END) AS flag_embedded_1,
       COUNT(vd.document_id) AS righe_vec_reali
FROM raw_documents rd
LEFT JOIN vec_documents vd ON vd.document_id = rd.id
GROUP BY rd.origin
""")
coerenza


,origin,flag_embedded_1,righe_vec_reali
0,comtrade,252,252
1,gdelt,174286,168544
2,rss,2972,2972


In [12]:
coerenza["drift"] = coerenza["flag_embedded_1"] - coerenza["righe_vec_reali"]
print("Drift = flag embedded=1 in raw_documents MENO righe reali in vec_documents (per origin).")
print("Drift != 0 indica documenti marcati embedded ma senza vettore scritto (o viceversa).")
coerenza


Drift = flag embedded=1 in raw_documents MENO righe reali in vec_documents (per origin).
Drift != 0 indica documenti marcati embedded ma senza vettore scritto (o viceversa).


,origin,flag_embedded_1,righe_vec_reali,drift
0,comtrade,252,252,0
1,gdelt,174286,168544,5742
2,rss,2972,2972,0


## 6. Quantificazione dello spreco (spazio/tempo)

Stima approssimativa: ogni vettore e5-small ha dimensione fissa (384 dim float32 = 1536 byte
raw, più overhead indice sqlite-vec). Il tempo di embed è proporzionale al numero di
documenti processati (batch throughput dell'embedder).

In [13]:
EMBED_DIM = 384
BYTES_PER_FLOAT = 4
bytes_per_vector = EMBED_DIM * BYTES_PER_FLOAT
spreco_bytes = vec_gdelt * bytes_per_vector
spreco_mb = spreco_bytes / (1024 * 1024)

print(f"Vettori gdelt 'sprecati' (titoli sintetici, non prosa): {vec_gdelt}")
print(f"Stima spazio: {vec_gdelt} x {EMBED_DIM} dim x {BYTES_PER_FLOAT} byte = "
      f"{spreco_bytes:,} byte ≈ {spreco_mb:.1f} MB di soli dati vettore "
      "(esclude overhead indice/chunk table, quindi è una stima per difetto).")
print()
print("Tempo: non misurabile direttamente da query SQL (dipende da throughput e5-small "
      "sull'hardware M1, non loggato per singolo embed nel DB) — nota qualitativa: "
      f"{vec_gdelt} embed su titoli sintetici sono state chiamate all'encoder e5-small "
      "completamente inutili dal punto di vista semantico (il testo è \"GDELT: ACTOR1 -> "
      "ACTOR2 [CODE]\", non prosa), tempo macchina bruciato per vettori che non verranno "
      "mai usati in ricerca semantica utile.")


Vettori gdelt 'sprecati' (titoli sintetici, non prosa): 168544
Stima spazio: 168544 x 384 dim x 4 byte = 258,883,584 byte ≈ 246.9 MB di soli dati vettore (esclude overhead indice/chunk table, quindi è una stima per difetto).

Tempo: non misurabile direttamente da query SQL (dipende da throughput e5-small sull'hardware M1, non loggato per singolo embed nel DB) — nota qualitativa: 168544 embed su titoli sintetici sono state chiamate all'encoder e5-small completamente inutili dal punto di vista semantico (il testo è "GDELT: ACTOR1 -> ACTOR2 [CODE]", non prosa), tempo macchina bruciato per vettori che non verranno mai usati in ricerca semantica utile.


## Sintesi criticità osservate in questo run

In [14]:
print("--- Sintesi (valori da questo run) ---")
print(f"1. Fix NON_PROSE_ORIGINS attivo e verificato: nuovi doc origin in ('gdelt','comtrade') "
      f"restano embedded=0. Snapshot attuale: gdelt embedded=0 -> "
      f"{int(contam.loc[contam['origin']=='gdelt','embedded_0_corretto'].sum()) if 'gdelt' in contam['origin'].values else 0}, "
      f"comtrade embedded=0 -> "
      f"{int(contam.loc[contam['origin']=='comtrade','embedded_0_corretto'].sum()) if 'comtrade' in contam['origin'].values else 0}.")
print(f"2. Contaminazione storica residua (embedded=1 pre-fix): {tot_contam}/{tot_all_non_prose} "
      f"({tot_contam/tot_all_non_prose*100:.1f}%) documenti gdelt/comtrade — il fix NON è "
      "retroattivo, serve backfill/cleanup esplicito per ripulire vec_documents.")
print(f"3. Vettori reali in vec_documents: {vec_total} totali, di cui {vec_gdelt} "
      f"({pct_gdelt_of_total:.1f}%) origin=gdelt, {vec_comtrade} origin=comtrade, {vec_rss} origin=rss.")
print(f"4. Spreco stimato spazio indice: ≈{spreco_mb:.1f} MB di vettori su titoli sintetici "
      "non-prosa, mai utili semanticamente.")
print("5. CONCLUSIONE: il fix impedisce nuova contaminazione ma NON pulisce quella esistente. "
      "Serve un job di backfill che: (a) individua vec_documents.document_id con "
      "raw_documents.origin IN ('gdelt','comtrade'), (b) li rimuove da vec_documents, "
      "(c) resetta embedded=0 su raw_documents corrispondenti per coerenza flag/tabella.")


--- Sintesi (valori da questo run) ---
1. Fix NON_PROSE_ORIGINS attivo e verificato: nuovi doc origin in ('gdelt','comtrade') restano embedded=0. Snapshot attuale: gdelt embedded=0 -> 2995, comtrade embedded=0 -> 0.
2. Contaminazione storica residua (embedded=1 pre-fix): 174538/177533 (98.3%) documenti gdelt/comtrade — il fix NON è retroattivo, serve backfill/cleanup esplicito per ripulire vec_documents.
3. Vettori reali in vec_documents: 171768 totali, di cui 168544 (98.1%) origin=gdelt, 252 origin=comtrade, 2972 origin=rss.
4. Spreco stimato spazio indice: ≈246.9 MB di vettori su titoli sintetici non-prosa, mai utili semanticamente.
5. CONCLUSIONE: il fix impedisce nuova contaminazione ma NON pulisce quella esistente. Serve un job di backfill che: (a) individua vec_documents.document_id con raw_documents.origin IN ('gdelt','comtrade'), (b) li rimuove da vec_documents, (c) resetta embedded=0 su raw_documents corrispondenti per coerenza flag/tabella.
